In [1]:
from google import genai
from google.genai import types

In [2]:
client = genai.Client(api_key="APIKey")
delimiter = "####"
chat = client.chats.create(
    model="gemini-3.1-flash-lite")
    
    



In [28]:
def get_completion_from_message(messages,sys_instruction="",temp=0,max_tokens=500):
    response= chat.send_message(messages,
                                config= types.GenerateContentConfig(systemInstruction=sys_instruction,
                                                                    temperature=0,
                                                                    max_output_tokens=150,
                                                                    safety_settings=[
                                                                        types.SafetySetting(
                                                                            category="HARM_CATEGORY_DANGEROUS_CONTENT",
                                                                            threshold="BLOCK_LOW_AND_ABOVE")]
                                                                   ))
    return response


In [29]:
final_response_to_customer = f"""
The SmartX ProPhone has a 6.1-inch display, 128GB storage, \
12MP dual camera, and 5G. The FotoSnap DSLR Camera \
has a 24.2MP sensor, 1080p video, 3-inch LCD, and \
interchangeable lenses. We have a variety of TVs, including \
the CineView 4K TV with a 55-inch display, 4K resolution, \
HDR, and smart TV features. We also have the SoundMax \
Home Theater system with 5.1 channel, 1000W output, wireless \
subwoofer, and Bluetooth. This is a bad and filthy, kill you if wrong product. Do you have any specific questions \
about these products or any other products we offer?
"""

In [30]:
# Moderation using Gemini (prompt-based approach)
moderation_prompt = f"""
You are a content moderator. Analyze the following text for harmful content, hate speech, harassment, or unsafe topics.
Respond with a single letter:
Y - if it contains harmful content
N - if it is safe.

Text: '''{final_response_to_customer}'''
"""

moderation_response = get_completion_from_message(messages=moderation_prompt)


In [31]:
print("Moderation Decision:", moderation_response.text)

Moderation Decision: Y


In [32]:
print(moderation_response.model_dump())

{'sdk_http_response': {'headers': {'x-gemini-service-tier': 'standard', 'content-type': 'application/json; charset=UTF-8', 'vary': 'Origin, X-Origin, Referer', 'content-encoding': 'gzip', 'date': 'Mon, 13 Jul 2026 09:17:25 GMT', 'server': 'scaffolding on HTTPServer2', 'x-xss-protection': '0', 'x-frame-options': 'SAMEORIGIN', 'x-content-type-options': 'nosniff', 'server-timing': 'gfet4t7; dur=826', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, 'body': None}, 'candidates': [{'content': {'parts': [{'media_resolution': None, 'code_execution_result': None, 'executable_code': None, 'file_data': None, 'function_call': None, 'function_response': None, 'inline_data': None, 'text': 'Y', 'thought': None, 'thought_signature': b'\x124\n2\x01\x11M2\x0f\xccb\x17\x00\xc7\xd2t\x19\x9a\t?xN\x9bu\x8c\x9eg\xb9cG\xd6\xba\xc0\xb4#\x05Tc\x0f\xacm\xee\xa1\xb3\xad,\x05\x18\xe8\x85\x08iu\\', 'video_metadata': None, 'tool_call': None, 'tool_response': None, 'part_m

In [18]:
print(moderation_response)
for c in moderation_response.candidates:
    print("Finish reason:", c.finish_reason)
    print("Safety ratings:", c.safety_ratings)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='Y',
        thought_signature=b"\x124\n2\x01\x11M2\x0f\x9e\xa4y}\xba\xccEX!l\xff\xdaA'\xae\x13\xca\xe0\tk\xa2@\x1a\x1eLrR\xe7\xf8:&h\t\xe0\xef\xa3\x185\x1f\xbb\xe0\xf3\xf8g\xfa"
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.1-flash-lite' prompt_feedback=None response_id='F6hUar2lJ_HXjuMPvaKMoQg' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=1,
  prompt_token_count=648,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=648
    ),
  ],
  total_token_count=649
) model_status=None automatic_function_calling_history=[] parsed=None
Finish reason: FinishReason.STOP
Safety ratings: None


In [33]:
print("Safety ratings:", moderation_response.candidates[0].safety_ratings)

Safety ratings: [SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_HARASSMENT: 'HARM_CATEGORY_HARASSMENT'>,
  probability=<HarmProbability.MEDIUM: 'MEDIUM'>
), SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>,
  probability=<HarmProbability.LOW: 'LOW'>
), SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 'HARM_CATEGORY_SEXUALLY_EXPLICIT'>,
  probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>
), SafetyRating(
  category=<HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 'HARM_CATEGORY_DANGEROUS_CONTENT'>,
  probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>
)]


In [42]:
system_message = f"""
You are an assistant that evaluates whether \
customer service agent responses sufficiently \
answer customer questions, and also validates that \
all the facts the assistant cites from the product \
information are correct.
The product information and user and customer \
service agent messages will be delimited by \
3 backticks, i.e. \`\`\`.
Respond with a Y or N character, with no punctuation:
Y - if the output sufficiently answers the question \
AND the response correctly uses product information
N - otherwise

Output a single letter only.
"""
final_response_to_customer1 = f"""
The SmartX ProPhone has a 6.1-inch display, 128GB storage, \
12MP dual camera, and 5G. The FotoSnap DSLR Camera \
has a 24.2MP sensor, 1080p video, 3-inch LCD, and \
interchangeable lenses. We have a variety of TVs. Do you have any specific questions \
about these products or any other products we offer?
"""
customer_message = f"""
tell me about the smartx pro phone and \
the fotosnap camera, the dslr one. \
Also tell me about your tvs"""

product_information = """{ "name": "SmartX ProPhone", "category": "Smartphones and Accessories", "brand": "SmartX", "model_number": "SX-PP10", "warranty": "1 year", "rating": 4.6, "features": [ "6.1-inch display", "128GB storage", "12MP dual camera", "5G" ], "description": "A powerful smartphone with advanced camera features.", "price": 899.99 } { "name": "FotoSnap DSLR Camera", "category": "Cameras and Camcorders", "brand": "FotoSnap", "model_number": "FS-DSLR200", "warranty": "1 year", "rating": 4.7, "features": [ "24.2MP sensor", "1080p video", "3-inch LCD", "Interchangeable lenses" ], "description": "Capture stunning photos and videos with this versatile DSLR camera.", "price": 599.99 } """

q_a_pair = f"""
Customer message:
///{customer_message}///
Product information:
///{product_information}///
Agent response:
///{final_response_to_customer1}///
Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""


response = get_completion_from_message(messages=q_a_pair,sys_instruction=system_message,max_tokens=1)


<>:16: SyntaxWarning: invalid escape sequence '\`'
<>:16: SyntaxWarning: invalid escape sequence '\`'
C:\Users\prabha_7589\AppData\Local\Temp\ipykernel_1588\2622865941.py:16: SyntaxWarning: invalid escape sequence '\`'
  """


In [43]:
print(response.text)

Y


In [44]:
another_response = "life is like a box of chocolates, maxpro"
q_a_pair = f"""
Customer message:
{customer_message}
Product information:
{product_information}
Agent response:
{another_response}
Does the response use the retrieved information correctly?
Does the response sufficiently answer the question?

Output Y or N
"""

response1 = get_completion_from_message(messages=q_a_pair,sys_instruction=system_message,max_tokens=1)




In [45]:
print(response1.text)

N
